# Laboratorio 13 — Ordenar sin particionar

**Duración: 45 minutos.**

En el laboratorio 04 aprendiste a particionar, y funcionó: de treinta mil filas leídas a
dos mil quinientas. Pero particionar no es gratis ni siempre se puede, y hoy vas a ver que
**no es la única palanca**.

La pregunta de hoy:

> Si no puedo particionar esta tabla, **¿estoy condenado a leerla entera?**

---

### En el almacén

Particionar es poner **pasillos rotulados**: todo lo de enero en un pasillo, todo lo de
febrero en otro. Para eso hay que elegir el rótulo, y a veces no hay uno bueno.

Ordenar es otra cosa. Es dejar las cajas **en orden de fecha** en un solo pasillo, sin
rótulos. Si alguien pregunta por junio, no hace falta abrir todas: con mirar por fuera la
primera y la última caja de cada montón, se sabe cuáles no pueden contener junio.

Eso funciona **si hay varios montones**. Con una sola caja gigante, ordenar por dentro no
sirve de nada. Ese es el primer descubrimiento de hoy.


## Paso 0 — Una tabla ordenada, sin particiones

Primero, a tu espacio de trabajo.

In [ ]:
%%sql
-- Celda 0.1



Partimos de cero.

In [ ]:
%%sql
-- Celda 0.2



Ahora la tabla del día. Fíjate bien en lo que tiene y en lo que no: lleva
`ORDER BY fecha_emision`, **pero no lleva `PARTITIONED BY`**. Un año completo de
documentos, ordenados por fecha, en una tabla sin particiones.

In [ ]:
%%sql
-- Celda 0.3



¿Cuántos quedaron?

In [ ]:
%%sql
-- Celda 0.4



Treinta mil, los de siempre. Y ahora la celda que decide el laboratorio: **en cuántos
archivos quedaron**, y qué rango de fechas abarca cada uno.

In [ ]:
%%sql
-- Celda 0.5



## Paso 1 — Partirla en varios archivos

**Un solo archivo, del 1 de enero al 31 de diciembre.**

Ahí está el problema, y es importante entenderlo antes de seguir: la tabla **está**
ordenada por dentro, pero eso no le sirve a nadie. El motor decide qué leer mirando
archivos enteros, y solo hay uno. Para responder por junio tiene que abrirlo, y con él
entra el año completo.

Ordenar sin tener varios archivos no hace nada. Vamos a partirla.

Para eso usamos el mismo `rewrite_data_files` del laboratorio 08, pero pidiéndole lo
contrario: allá juntaba archivos chicos en uno grande, aquí le decimos que apunte a
archivos de unos 250 kilobytes y que ordene por fecha al reescribir.

In [ ]:
%%sql
-- Celda 1.1



Y ahora mira otra vez los archivos y sus rangos.

In [ ]:
%%sql
-- Celda 1.2



## Paso 2 — La regla de medir

**Nueve archivos, cada uno con su tramo del año**, del más antiguo al más nuevo y sin
pisarse salvo en el día del borde. Eso es lo que hace el orden: agrupa por **rango**.

Ahora la regla de medir del laboratorio 04, la misma consulta, sobre esta tabla. Cuenta
cuántos archivos tendría que abrir el motor para responder por junio.

In [ ]:
%%sql
-- Celda 2.1



## Paso 3 — El cuadro completo

**Dos archivos y unas ocho mil filas**, contra los treinta mil de antes. Sin una sola
partición.

Este es el cuadro que cierra la primera mitad del laboratorio, y conviene copiarlo tal
cual:

| Tabla | Archivos a leer | Filas a leer |
|---|---|---|
| Sin nada, un archivo | 1 | 30.000 |
| **Ordenada por fecha, nueve archivos** | **2** | **~8.000** |
| Particionada por mes, laboratorio 04 | 1 | 2.500 |

Léelo de arriba abajo. Ordenar no llega a lo que consigue particionar, y era esperable:
un rango de fechas es más grueso que un rótulo exacto. Pero pasa de leer el año entero a
leer dos meses, **sin declarar ninguna partición**.

Y eso importa cuando particionar no es una opción: cuando no hay una columna con pocos
valores distintos, cuando el criterio de consulta cambia seguido, o cuando particionar
generaría tantos archivos chicos que el remedio sería peor.

## Paso 4 — Lo que pasa cuando llega una carga desordenada

Hasta aquí todo bien. Pero el orden es frágil, y esto es lo que hay que saber antes de
apoyarse en él.

Llega una carga nueva. Los mismos documentos del año, pero **en orden aleatorio**, como
llegaría de un sistema que no ordena nada.

Fíjate en que es **el mismo año otra vez**: la tabla va a quedar con sesenta mil documentos
y cada uno repetido dos veces. Es a propósito, y no importa para lo de hoy, porque lo que
vamos a mirar es **cómo quedan repartidos los archivos**, no qué dice cada fila. Lo único
que necesitamos de esta carga es que llegue desordenada.

In [ ]:
%%sql
-- Celda 4.1



Mira los archivos otra vez.

In [ ]:
%%sql
-- Celda 4.2



**Ahí está el daño.** Hay un archivo nuevo que abarca del 1 de enero al 31 de diciembre,
porque sus filas llegaron mezcladas. Ese archivo va a entrar en **todas** las consultas,
pregunten por el mes que pregunten.

Vuelve a medir para junio.

In [ ]:
%%sql
-- Celda 4.3



## Paso 5 — Reordenar lo que ya está

Subió, y subió mucho: el archivo desordenado se suma a cualquier consulta.

La buena noticia es que esto se arregla, y con la misma herramienta del paso 1. No hay que
recargar nada ni recrear la tabla: se reordena **lo que ya está escrito**.

In [ ]:
%%sql
-- Celda 5.1



Los archivos, por tercera vez.

In [ ]:
%%sql
-- Celda 5.2



**Otra vez rangos ordenados**, ahora con sesenta mil documentos repartidos. Mide de
nuevo y compara con lo que tenías después del paso 4.

In [ ]:
%%sql
-- Celda 5.3



Y una última mirada a la libreta, para ver qué dejó escrito la reorganización.

In [ ]:
%%sql
-- Celda 5.4



## Cierre

Dos palancas para lo mismo, y no son excluyentes:

| | Agrupa por | Cuándo sirve |
|---|---|---|
| **Particionar** | valor exacto, `mes = 2024-06` | hay una columna con pocos valores distintos y estable |
| **Ordenar** | rango, `del 2 de mayo al 19 de junio` | no hay buen criterio de partición, o cambia |

Las dos hacen lo mismo en el fondo: que el motor **pueda descartar archivos sin abrirlos**.
Ese es el mecanismo, y es el mismo que viste en el laboratorio 04 con los rótulos.

Tres cosas que te llevas:

- **Ordenar sin varios archivos no hace nada.** Con un archivo gigante da igual lo ordenado
  que esté por dentro.
- **El orden se degrada solo.** Cada carga desordenada agrega un archivo que abarca todo y
  se mete en todas las consultas. No es un error, es cómo funciona.
- **Se repara sin recargar.** `rewrite_data_files` con `sort_order` reordena lo escrito, y
  es una rutina de mantenimiento como las del laboratorio 08.

Y se combinan. Una tabla puede estar particionada por mes **y** ordenada por RUT dentro de
cada partición. Ahí el motor descarta por mes y después descarta por rango, y lee todavía
menos.
